# Submission 

In [1]:
# ============================================================
# ONE CELL — SUBMISSION (FULL REVISI, ARCH + FORMAT SAFE) — vFinal
# - Auto-detect FINAL_MODEL_PT (fix path mismatch model_bundle_v3 vs root artifacts)
# - Compatible final_gate_model.pt formats:
#     (A) {"packs":[{state_dict,mu,sig,cfg}, ...]}                           (legacy)
#     (B) {"fold_packs":[...], "full_packs":[...], "recommended_thr": ...}   (Step5 v4+)
#     (C) {"feature_cols":[...], ...}  (prefer feature_cols inside pt)
# - Supports arch by state_dict keys:
#     * FTTransformer_MHCLite  : has "mhc." or "out_norm." (Step4/5 transformer gate)
#     * TransformerEncoder     : has "encoder.layers."
#     * Custom blocks variant  : has "final_norm." / "blocks."
# - pred_features_test*.csv NOT required (auto-search). If missing -> features become zeros (warn).
# - Optional mask source: pred_ens/{case_id}.npz (auto-search). If missing -> forged falls back to "authentic".
# Output: /kaggle/working/submission.csv
# ============================================================

import os, json, time, gc, math, inspect, warnings
from pathlib import Path

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings("ignore")

# optional (for faster nearest resize); fallback to PIL if absent
try:
    import cv2
    _HAS_CV2 = True
except Exception:
    _HAS_CV2 = False

try:
    from IPython.display import display
except Exception:
    def display(x): 
        print(x)

# ----------------------------
# USER CONFIG (hint only; auto-detect if wrong)
# ----------------------------
FINAL_MODEL_PT_HINT = Path("/kaggle/input/recod-ailuc-dinov2-train/recodai_luc_gate_artifacts/model_bundle_v3/final_gate_model.pt")
AGG_MODE = "mean"     # "mean" / "max" / "p80"
BATCH_SIZE = 4096     # auto-fallback on OOM (cuda)
FORCE_T_GATE = None   # set float to override thresholds.json / recommended_thr, e.g. 0.4706

# ----------------------------
# 0) Auto-find FINAL_MODEL_PT (fix: dataset kamu simpan pt di root artifacts)
# ----------------------------
def _score_model_pt(p: Path):
    p = Path(p)
    d = p.parent
    sc = 0
    if d.name == "recodai_luc_gate_artifacts": sc += 100
    if (d / "feature_cols.json").exists(): sc += 20
    if (d / "final_gate_bundle.json").exists(): sc += 10
    if (d / "thresholds.json").exists(): sc += 5
    if any(d.glob("pred_features_test*.csv")): sc += 2
    return sc

def _auto_find_final_model_pt(user_hint: Path | None):
    # 1) hint path
    if user_hint is not None and Path(user_hint).exists():
        return Path(user_hint)

    cands = []

    # 2) common working path
    p = Path("/kaggle/working/recodai_luc_gate_artifacts/final_gate_model.pt")
    if p.exists():
        cands.append(p)

    # 3) common input pattern
    base = Path("/kaggle/input")
    if base.exists():
        for ds in base.iterdir():
            if not ds.is_dir():
                continue
            p = ds / "recodai_luc_gate_artifacts" / "final_gate_model.pt"
            if p.exists():
                cands.append(p)

    # 4) limited fallback search (only likely datasets)
    if base.exists() and not cands:
        for ds in base.iterdir():
            if not ds.is_dir():
                continue
            if not any(k in ds.name.lower() for k in ["recod", "luc", "dinov2", "gate", "ailuc"]):
                continue
            hits = list(ds.rglob("final_gate_model.pt"))
            if hits:
                cands.extend(hits[:10])

    if not cands:
        raise FileNotFoundError(
            "final_gate_model.pt tidak ditemukan di /kaggle/working maupun /kaggle/input.\n"
            "Pastikan dataset artifacts sudah di-add ke notebook ini."
        )

    cands = sorted(cands, key=lambda p: (_score_model_pt(p), -len(str(p))), reverse=True)
    return cands[0]

FINAL_MODEL_PT = _auto_find_final_model_pt(FINAL_MODEL_PT_HINT)

if not FINAL_MODEL_PT.exists():
    raise FileNotFoundError(f"Model not found: {FINAL_MODEL_PT}")

BUNDLE_DIR = FINAL_MODEL_PT.parent
ART_DIR = BUNDLE_DIR if BUNDLE_DIR.name == "recodai_luc_gate_artifacts" else BUNDLE_DIR.parent

print("Using:")
print("  FINAL_MODEL_PT:", FINAL_MODEL_PT)
print("  BUNDLE_DIR    :", BUNDLE_DIR)
print("  ART_DIR       :", ART_DIR)

# ----------------------------
# 1) Competition paths (auto-detect)
# ----------------------------
def find_comp_root():
    pref = Path("/kaggle/input/recodai-luc-scientific-image-forgery-detection")
    if pref.exists():
        return pref
    base = Path("/kaggle/input")
    cands = []
    for d in base.iterdir():
        if d.is_dir() and (d / "sample_submission.csv").exists() and (d / "test_images").exists():
            cands.append(d)
    if not cands:
        raise FileNotFoundError("Competition folder not found under /kaggle/input")
    cands.sort(key=lambda x: (("recod" not in x.name.lower()), x.name))
    return cands[0]

COMP_ROOT = find_comp_root()
SAMPLE_SUB = COMP_ROOT / "sample_submission.csv"
TEST_IMG_DIR = COMP_ROOT / "test_images"
if not SAMPLE_SUB.exists():
    raise FileNotFoundError(f"Missing sample_submission.csv: {SAMPLE_SUB}")

print("  COMP_ROOT   :", COMP_ROOT)
print("  SAMPLE_SUB  :", SAMPLE_SUB)
print("  TEST_IMG_DIR:", TEST_IMG_DIR)

# ----------------------------
# 2) Robust torch.load (PyTorch 2.6 weights_only default True)
# ----------------------------
def torch_load_robust(path, map_location="cpu"):
    path = str(path)
    sig = None
    try:
        sig = inspect.signature(torch.load)
    except Exception:
        sig = None

    if sig is not None and "weights_only" in sig.parameters:
        try:
            return torch.load(path, map_location=map_location, weights_only=True)
        except Exception:
            return torch.load(path, map_location=map_location, weights_only=False)
    return torch.load(path, map_location=map_location)

# ----------------------------
# 3) Find nearby artifacts (feature_cols.json, thresholds.json)
# ----------------------------
def find_nearby(filename: str, roots):
    roots = [Path(r) for r in roots]
    for r in roots:
        p = r / filename
        if p.exists():
            return p
    for r in roots:
        if r.exists():
            hits = list(r.glob(f"**/{filename}"))
            if hits:
                hits.sort(key=lambda x: len(str(x)))
                return hits[0]
    return None

# ----------------------------
# 4) Load model blob + extract packs (supports many formats)
# ----------------------------
blob = torch_load_robust(FINAL_MODEL_PT, map_location="cpu")
if not isinstance(blob, dict):
    raise ValueError(f"Unexpected final_gate_model.pt type={type(blob)} (expected dict)")

# Feature cols priority:
#   1) blob["feature_cols"] (new)
#   2) feature_cols.json near bundle (old)
FEATURE_COLS = None
if isinstance(blob.get("feature_cols", None), list) and len(blob["feature_cols"]) > 0:
    FEATURE_COLS = list(blob["feature_cols"])
    FEATURE_COLS_JSON = None
else:
    FEATURE_COLS_JSON = find_nearby(
        "feature_cols.json",
        [BUNDLE_DIR, ART_DIR, Path("/kaggle/working/recodai_luc_gate_artifacts")]
    )
    if FEATURE_COLS_JSON is None:
        raise FileNotFoundError("feature_cols.json not found and blob has no feature_cols.")
    FEATURE_COLS = json.loads(FEATURE_COLS_JSON.read_text())

if not isinstance(FEATURE_COLS, list) or len(FEATURE_COLS) == 0:
    raise ValueError("FEATURE_COLS invalid/empty.")

# Packs priority:
#   1) fold_packs
#   2) full_packs
#   3) packs (legacy)
packs = None
pack_src = None
if isinstance(blob.get("fold_packs", None), list) and len(blob["fold_packs"]) > 0:
    packs = blob["fold_packs"]
    pack_src = "blob.fold_packs"
elif isinstance(blob.get("full_packs", None), list) and len(blob["full_packs"]) > 0:
    packs = blob["full_packs"]
    pack_src = "blob.full_packs"
elif isinstance(blob.get("packs", None), list) and len(blob["packs"]) > 0:
    packs = blob["packs"]
    pack_src = "blob.packs(legacy)"

if packs is None:
    raise ValueError(f"No packs found in model. keys={list(blob.keys())}")

print("Artifacts:")
print("  feature_cols:", (FEATURE_COLS_JSON if FEATURE_COLS_JSON is not None else "from final_gate_model.pt"))
print("  pack_source :", pack_src)
print("  n_features  :", len(FEATURE_COLS))
print("  n_packs     :", len(packs))

# thresholds priority:
#   (a) thresholds.json near
#   (b) blob["recommended_thr"]
#   (c) 0.5
THR_JSON = find_nearby(
    "thresholds.json",
    [BUNDLE_DIR, ART_DIR, Path("/kaggle/working/recodai_luc_gate_artifacts")]
)

T_GATE = None
if THR_JSON is not None and THR_JSON.exists():
    try:
        thr = json.loads(THR_JSON.read_text())
        if isinstance(thr, dict):
            for k in ["T_gate", "gate_thr", "best_thr", "oof_best_thr", "recommended_thr"]:
                if k in thr and thr[k] is not None:
                    T_GATE = float(thr[k])
                    break
    except Exception:
        T_GATE = None

if T_GATE is None and blob.get("recommended_thr", None) is not None:
    try:
        T_GATE = float(blob["recommended_thr"])
    except Exception:
        T_GATE = None

if T_GATE is None:
    T_GATE = 0.5

if FORCE_T_GATE is not None:
    T_GATE = float(FORCE_T_GATE)

print("  thresholds  :", (THR_JSON if THR_JSON else "(not found)"))
print("  T_GATE      :", T_GATE)

# ----------------------------
# 5) Build base test table (case_id list)
# ----------------------------
df_sub = pd.read_csv(SAMPLE_SUB)
if "case_id" not in df_sub.columns:
    raise ValueError(f"sample_submission missing case_id. cols={list(df_sub.columns)}")
df_sub["case_id"] = df_sub["case_id"].astype(str)

df_base = pd.DataFrame({"case_id": df_sub["case_id"].values})
print("Base test cases:", df_base.shape[0])

# ----------------------------
# 6) Auto-find test feature tables (pred/match) (optional) + SAFE AGG (no row explosion)
# ----------------------------
def iter_feature_search_roots():
    # write outputs
    yield Path("/kaggle/working/recodai_luc_gate_artifacts")
    yield Path("/kaggle/working/recodai_luc/cache")
    yield Path("/kaggle/working")
    # bundle vicinity
    yield ART_DIR
    yield BUNDLE_DIR
    # inputs that look related
    base = Path("/kaggle/input")
    if base.exists():
        for d in base.iterdir():
            if d.is_dir() and any(t in d.name.lower() for t in ["recod", "luc", "dinov2", "gate", "bundle", "ailuc"]):
                yield d

def _read_table_any(p: Path):
    p = Path(p)
    if p.suffix.lower() == ".parquet":
        return pd.read_parquet(p)
    return pd.read_csv(p)

def _peek_cols(p: Path, nrows=8):
    p = Path(p)
    try:
        if p.suffix.lower() == ".parquet":
            dfh = pd.read_parquet(p).head(nrows)
        else:
            dfh = pd.read_csv(p, nrows=nrows)
        return list(dfh.columns)
    except Exception:
        return None

def _score_feature_file(p: Path, feature_cols: list):
    cols = _peek_cols(p)
    if not cols:
        return -1
    cols_set = set(cols)
    score = 0
    if "case_id" in cols_set: score += 2000
    if "uid" in cols_set:     score += 800
    if "variant" in cols_set: score += 50
    overlap = len(cols_set.intersection(set(feature_cols)))
    score += overlap
    # prefer cfg-specific or shorter path
    if "cfg_" in p.name: score += 5
    return score

def find_best_feature_file(patterns, feature_cols):
    cands = []
    for root in iter_feature_search_roots():
        if not root.exists():
            continue
        for pat in patterns:
            try:
                for p in root.rglob(pat):
                    if p.is_file():
                        cands.append(p)
            except Exception:
                pass
        if len(cands) >= 600:
            break
    if not cands:
        return None
    scored = []
    for p in cands:
        sc = _score_feature_file(p, feature_cols)
        if sc > 0:
            scored.append((sc, p))
    if not scored:
        return None
    scored.sort(reverse=True, key=lambda x: (x[0], -len(str(x[1]))))
    return scored[0][1]

def ensure_case_id(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    if "case_id" not in df.columns:
        if "uid" in df.columns:
            df["case_id"] = df["uid"].astype(str).str.extract(r"^(\d+)")[0]
        elif "id" in df.columns:
            df["case_id"] = df["id"].astype(str)
        else:
            raise ValueError("Cannot infer case_id from feature table.")
    df["case_id"] = df["case_id"].astype(str)
    return df

PRED_FEAT_ANY  = find_best_feature_file(["pred_features_test*.csv", "pred_features_test*.parquet"], FEATURE_COLS)
MATCH_FEAT_ANY = find_best_feature_file(["match_features_test*.csv", "match_features_test*.parquet"], FEATURE_COLS)

print("Feature candidates:")
print("  pred_features_test* :", (PRED_FEAT_ANY if PRED_FEAT_ANY else "(not found)"))
print("  match_features_test*:", (MATCH_FEAT_ANY if MATCH_FEAT_ANY else "(not found)"))

df_feat = df_base.copy()

def merge_features_agg(df_feat, feat_path: Path, tag: str):
    if feat_path is None:
        return df_feat
    try:
        dfx = _read_table_any(feat_path)
        dfx = ensure_case_id(dfx)

        keep_feats = [c for c in FEATURE_COLS if c in dfx.columns]
        keep_cols = ["case_id"] + keep_feats
        if "variant" in dfx.columns and "variant" not in keep_cols:
            keep_cols.append("variant")
        dfx = dfx[keep_cols].copy()

        # if duplicates per case_id -> aggregate numeric features to avoid row explosion
        if dfx["case_id"].duplicated().any():
            agg_dict = {c: "mean" for c in keep_feats}
            dfx = dfx.groupby("case_id", as_index=False).agg(agg_dict)
            print(f"INFO {tag}: duplicates detected -> aggregated by mean per case_id")

        before = len(df_feat)
        df_feat2 = df_feat.merge(dfx, on="case_id", how="left")
        after = len(df_feat2)
        print(f"Merged {tag}: {feat_path} | kept_feats={len(keep_feats)} | rows {before}->{after}")
        return df_feat2
    except Exception as e:
        print(f"WARN: failed load/merge {tag}: {feat_path} | err={repr(e)}")
        return df_feat

df_feat = merge_features_agg(df_feat, PRED_FEAT_ANY, "pred")
df_feat = merge_features_agg(df_feat, MATCH_FEAT_ANY, "match")

# ensure all required columns exist
for c in FEATURE_COLS:
    if c not in df_feat.columns:
        df_feat[c] = 0.0

# coerce numeric + nan/inf -> 0
for c in FEATURE_COLS:
    df_feat[c] = pd.to_numeric(df_feat[c], errors="coerce")

X_test = df_feat[FEATURE_COLS].to_numpy(dtype=np.float32, copy=True)
if not np.isfinite(X_test).all():
    X_test = np.nan_to_num(X_test, nan=0.0, posinf=0.0, neginf=0.0)

nonzero_rate = float(np.mean(np.abs(X_test).sum(axis=1) > 0)) * 100.0
print(f"Feature coverage sanity: rows_with_any_nonzero_feature = {nonzero_rate:.2f}%")
if nonzero_rate < 1.0:
    print("WARN: Hampir semua fitur nol. pred/match_features_test kemungkinan belum ada. Gate jadi tidak informatif.")

# ----------------------------
# 7) Inference: model defs (MHCLite / encoder / blocks)
# ----------------------------
class XDataset(Dataset):
    def __init__(self, X):
        self.X = torch.from_numpy(X.astype(np.float32))
    def __len__(self): return self.X.shape[0]
    def __getitem__(self, i): return self.X[i]

def apply_standardizer(X_in: np.ndarray, mu, sig):
    mu = np.asarray(mu, dtype=np.float32).reshape(-1)
    sig = np.asarray(sig, dtype=np.float32).reshape(-1)
    if mu.size != X_in.shape[1] or sig.size != X_in.shape[1]:
        mu = np.zeros((X_in.shape[1],), dtype=np.float32)
        sig = np.ones((X_in.shape[1],), dtype=np.float32)
    sig = np.where(sig < 1e-8, 1.0, sig).astype(np.float32)
    return ((X_in - mu) / sig).astype(np.float32)

def normalize_state_dict(sd: dict):
    if not isinstance(sd, dict):
        return sd
    for pref in ["module.", "model."]:
        if len(sd) > 0 and all(str(k).startswith(pref) for k in sd.keys()):
            sd = {str(k)[len(pref):]: v for k, v in sd.items()}
    if any(str(k).startswith("module.") for k in sd.keys()):
        sd = {(str(k)[7:] if str(k).startswith("module.") else str(k)): v for k, v in sd.items()}
    return sd

def detect_arch(sd: dict):
    keys = list(sd.keys())
    if any(k.startswith("mhc.") for k in keys) or any(k.startswith("out_norm.") for k in keys) or any(str(k).endswith("alpha_logit") for k in keys):
        return "mhc_lite"
    if any(k.startswith("encoder.layers.") for k in keys):
        return "encoder"
    if any(k.startswith("blocks.") for k in keys) and any(k.startswith("final_norm.") for k in keys):
        return "blocks_final_norm"
    if any(k.startswith("blocks.") for k in keys):
        return "blocks_final_norm"
    return "encoder"

# ---- Encoder variant (encoder.layers.*) ----
class FTTransformerEncoder(nn.Module):
    def __init__(self, n_features, d_model, n_heads, n_layers, ffn_mult, dropout, attn_dropout):
        super().__init__()
        self.w = nn.Parameter(torch.randn(n_features, d_model) * 0.02)
        self.b = nn.Parameter(torch.zeros(n_features, d_model))
        self.feat_emb = nn.Parameter(torch.randn(n_features, d_model) * 0.02)
        self.cls = nn.Parameter(torch.randn(1, 1, d_model) * 0.02)

        enc_layer = nn.TransformerEncoderLayer(
            d_model=int(d_model),
            nhead=int(n_heads),
            dim_feedforward=int(ffn_mult * d_model),
            dropout=float(dropout),
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=int(n_layers))
        self.token_dropout = nn.Dropout(float(attn_dropout))
        self.norm = nn.LayerNorm(int(d_model))

        self.head = nn.Sequential(
            nn.Linear(int(d_model), int(d_model)),
            nn.GELU(),
            nn.Dropout(float(dropout)),
            nn.Linear(int(d_model), 1),
        )

    def forward(self, x):
        tok = x.unsqueeze(-1) * self.w.unsqueeze(0) + self.b.unsqueeze(0)   # (B,F,D)
        tok = tok + self.feat_emb.unsqueeze(0)
        tok = self.token_dropout(tok)
        B = tok.size(0)
        cls = self.cls.expand(B, -1, -1)
        seq = torch.cat([cls, tok], dim=1)
        z = self.encoder(seq)
        z = self.norm(z[:, 0])
        return self.head(z).squeeze(-1)

# ---- Blocks + final_norm variant (older) ----
class RMSNormScale(nn.Module):
    def __init__(self, d_model, eps=1e-6):
        super().__init__()
        self.scale = nn.Parameter(torch.ones(int(d_model)))
        self.eps = float(eps)
    def forward(self, x):
        rms = torch.sqrt(torch.mean(x * x, dim=-1, keepdim=True) + self.eps)
        return (x / rms) * self.scale

class BlockFinalNorm(nn.Module):
    def __init__(self, d_model, n_heads, ffn_mult, dropout, attn_dropout):
        super().__init__()
        d_model = int(d_model)
        self.norm1 = RMSNormScale(d_model)
        self.attn = nn.MultiheadAttention(d_model, int(n_heads), dropout=float(attn_dropout), batch_first=True)
        self.norm2 = RMSNormScale(d_model)
        ffn_dim = int(ffn_mult * d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, ffn_dim),
            nn.GELU(),
            nn.Dropout(float(dropout)),
            nn.Linear(ffn_dim, d_model),
        )
        self.drop = nn.Dropout(float(dropout))

    def forward(self, x):
        h = self.norm1(x)
        a, _ = self.attn(h, h, h, need_weights=False)
        x = x + self.drop(a)
        h = self.norm2(x)
        x = x + self.drop(self.ffn(h))
        return x

class FTTransformerBlocksFinalNorm(nn.Module):
    def __init__(self, n_features, d_model, n_heads, n_layers, ffn_mult, dropout, attn_dropout):
        super().__init__()
        d_model = int(d_model)
        self.w = nn.Parameter(torch.randn(n_features, d_model) * 0.02)
        self.b = nn.Parameter(torch.zeros(n_features, d_model))
        self.feat_emb = nn.Parameter(torch.randn(n_features, d_model) * 0.02)
        self.cls = nn.Parameter(torch.randn(1, 1, d_model) * 0.02)

        self.token_dropout = nn.Dropout(float(attn_dropout))
        self.blocks = nn.ModuleList([
            BlockFinalNorm(d_model, n_heads, ffn_mult, dropout, attn_dropout) for _ in range(int(n_layers))
        ])
        self.final_norm = RMSNormScale(d_model)
        self.head = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.GELU(),
            nn.Dropout(float(dropout)),
            nn.Linear(d_model, 1),
        )

    def forward(self, x):
        tok = x.unsqueeze(-1) * self.w.unsqueeze(0) + self.b.unsqueeze(0)
        tok = tok + self.feat_emb.unsqueeze(0)
        tok = self.token_dropout(tok)
        B = tok.size(0)
        cls = self.cls.expand(B, -1, -1)
        z = torch.cat([cls, tok], dim=1)
        for blk in self.blocks:
            z = blk(z)
        z = self.final_norm(z)[:, 0]
        return self.head(z).squeeze(-1)

# ---- MHCLite gate (Step4/5) ----
class RMSNorm(nn.Module):
    def __init__(self, d, eps=1e-6):
        super().__init__()
        self.eps = float(eps)
        self.weight = nn.Parameter(torch.ones(int(d)))
    def forward(self, x):
        rms = torch.mean(x * x, dim=-1, keepdim=True)
        x = x * torch.rsqrt(rms + self.eps)
        return x * self.weight

def sinkhorn_knopp(P, tmax=20, eps=1e-6):
    M = P.clamp_min(eps)
    for _ in range(int(tmax)):
        M = M / (M.sum(dim=-1, keepdim=True).clamp_min(eps))
        M = M / (M.sum(dim=-2, keepdim=True).clamp_min(eps))
    return M

class MHCLite(nn.Module):
    def __init__(self, d_model, n_streams=4, tmax=20, dropout=0.0):
        super().__init__()
        self.n = int(n_streams)
        self.tmax = int(tmax)
        self.drop = nn.Dropout(float(dropout))
        self.norm = RMSNorm(d_model, eps=1e-6)
        self.mlp = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.GELU(),
            nn.Linear(d_model, self.n * self.n),
        )
        self.softplus = nn.Softplus()
        a0 = 0.01
        self.alpha_logit = nn.Parameter(torch.log(torch.tensor(a0 / (1 - a0), dtype=torch.float32)))

    def forward(self, streams, cls_vec):
        B, n, D = streams.shape
        h = self.norm(cls_vec)
        logits = self.mlp(h).view(B, n, n)
        P = self.softplus(logits)
        M = sinkhorn_knopp(P, tmax=self.tmax, eps=1e-6)

        alpha = torch.sigmoid(self.alpha_logit).to(dtype=streams.dtype, device=streams.device)
        I = torch.eye(n, device=streams.device, dtype=streams.dtype).unsqueeze(0).expand(B, -1, -1)
        H = (1.0 - alpha) * I + alpha * M

        mixed = torch.einsum("bij,bjd->bid", H, streams)
        injected = mixed + cls_vec.unsqueeze(1)
        return self.drop(injected)

class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, ffn_mult=4, dropout=0.2, attn_dropout=0.1):
        super().__init__()
        self.norm1 = RMSNorm(d_model, eps=1e-6)
        self.attn = nn.MultiheadAttention(embed_dim=int(d_model), num_heads=int(n_heads),
                                          dropout=float(attn_dropout), batch_first=True)
        self.drop1 = nn.Dropout(float(dropout))
        self.norm2 = RMSNorm(d_model, eps=1e-6)
        self.ffn = nn.Sequential(
            nn.Linear(int(d_model), int(ffn_mult) * int(d_model)),
            nn.GELU(),
            nn.Dropout(float(dropout)),
            nn.Linear(int(ffn_mult) * int(d_model), int(d_model)),
        )
        self.drop2 = nn.Dropout(float(dropout))

    def forward(self, x):
        h = self.norm1(x)
        attn_out, _ = self.attn(h, h, h, need_weights=False)
        x = x + self.drop1(attn_out)
        h = self.norm2(x)
        x = x + self.drop2(self.ffn(h))
        return x

class FTTransformer_MHCLite(nn.Module):
    def __init__(self, n_features, d_model=384, n_heads=8, n_layers=8, ffn_mult=4,
                 dropout=0.2, attn_dropout=0.1,
                 n_streams=4, sinkhorn_tmax=20, mhc_dropout=0.0):
        super().__init__()
        self.n_features = int(n_features)
        self.d_model = int(d_model)
        self.n_layers = int(n_layers)

        self.w = nn.Parameter(torch.randn(self.n_features, self.d_model) * 0.02)
        self.b = nn.Parameter(torch.zeros(self.n_features, self.d_model))
        self.feat_emb = nn.Parameter(torch.randn(self.n_features, self.d_model) * 0.02)

        self.cls = nn.Parameter(torch.randn(1, 1, self.d_model) * 0.02)
        self.in_drop = nn.Dropout(float(dropout))

        self.blocks = nn.ModuleList([
            TransformerBlock(self.d_model, n_heads, ffn_mult=ffn_mult, dropout=dropout, attn_dropout=attn_dropout)
            for _ in range(self.n_layers)
        ])
        self.mhc = nn.ModuleList([
            MHCLite(self.d_model, n_streams=n_streams, tmax=sinkhorn_tmax, dropout=mhc_dropout)
            for _ in range(self.n_layers)
        ])
        self.out_norm = RMSNorm(self.d_model, eps=1e-6)
        self.head = nn.Sequential(
            nn.Linear(self.d_model, self.d_model),
            nn.GELU(),
            nn.Dropout(float(dropout)),
            nn.Linear(self.d_model, 1),
        )

    def forward(self, x):
        tok = x.unsqueeze(-1) * self.w.unsqueeze(0) + self.b.unsqueeze(0)
        tok = tok + self.feat_emb.unsqueeze(0)

        B = tok.size(0)
        cls = self.cls.expand(B, -1, -1)
        seq = torch.cat([cls, tok], dim=1)
        seq = self.in_drop(seq)

        nS = self.mhc[0].n
        streams = seq[:, 0, :].unsqueeze(1).expand(B, nS, self.d_model).contiguous()

        for l, blk in enumerate(self.blocks):
            cls_in = streams.mean(dim=1).unsqueeze(1)
            seq = torch.cat([cls_in, seq[:, 1:, :]], dim=1)
            seq = blk(seq)
            cls_vec = seq[:, 0, :]
            streams = self.mhc[l](streams, cls_vec)

        out = self.out_norm(streams.mean(dim=1))
        logit = self.head(out).squeeze(-1)
        return logit

def infer_ffn_mult_from_sd(sd, d_model, default=4):
    k = "blocks.0.ffn.0.weight"
    if k in sd and hasattr(sd[k], "shape"):
        try:
            return int(sd[k].shape[0] // int(d_model))
        except Exception:
            pass
    return int(default)

def infer_n_layers_from_sd(sd, prefix="blocks."):
    idxs = []
    for k in sd.keys():
        ks = str(k)
        if ks.startswith(prefix):
            sp = ks.split(".")
            if len(sp) >= 2:
                try:
                    idxs.append(int(sp[1]))
                except Exception:
                    pass
    return (max(idxs) + 1) if idxs else None

def infer_n_streams_from_sd(sd, default=4):
    k = "mhc.0.mlp.2.weight"
    if k in sd and hasattr(sd[k], "shape"):
        try:
            out_dim = int(sd[k].shape[0])
            n = int(round(math.sqrt(out_dim)))
            if n * n == out_dim:
                return n
        except Exception:
            pass
    return int(default)

def build_model_for_pack(pack_state_dict: dict, n_features: int, cfg: dict):
    sd = normalize_state_dict(pack_state_dict)
    arch = detect_arch(sd)

    d_model = int(cfg.get("d_model", 384)) if isinstance(cfg, dict) else 384
    if "w" in sd and hasattr(sd["w"], "shape"):
        try:
            d_model = int(sd["w"].shape[1])
        except Exception:
            pass

    n_layers = int(cfg.get("n_layers", 8)) if isinstance(cfg, dict) else 8
    nl_sd = infer_n_layers_from_sd(sd, prefix="blocks.")
    if nl_sd is not None:
        n_layers = int(nl_sd)
    if arch == "encoder":
        idxs = []
        for k in sd.keys():
            ks = str(k)
            if ks.startswith("encoder.layers."):
                try:
                    idxs.append(int(ks.split(".")[2]))
                except Exception:
                    pass
        if idxs:
            n_layers = max(idxs) + 1

    ffn_mult = int(cfg.get("ffn_mult", 4)) if isinstance(cfg, dict) else 4
    ffn_mult = infer_ffn_mult_from_sd(sd, d_model, default=ffn_mult)

    n_heads = int(cfg.get("n_heads", 8)) if isinstance(cfg, dict) else 8
    if int(d_model) % int(n_heads) != 0:
        for cand in [16, 8, 4, 2, 1]:
            if int(d_model) % cand == 0:
                n_heads = cand
                break

    dropout = float(cfg.get("dropout", 0.2)) if isinstance(cfg, dict) else 0.2
    attn_dropout = float(cfg.get("attn_dropout", 0.1)) if isinstance(cfg, dict) else 0.1

    if arch == "mhc_lite":
        n_streams = int(cfg.get("n_streams", 4)) if isinstance(cfg, dict) else 4
        n_streams = infer_n_streams_from_sd(sd, default=n_streams)
        sinkhorn_tmax = int(cfg.get("sinkhorn_tmax", 20)) if isinstance(cfg, dict) else 20
        mhc_dropout = float(cfg.get("mhc_dropout", 0.0)) if isinstance(cfg, dict) else 0.0
        m = FTTransformer_MHCLite(
            n_features=n_features, d_model=d_model, n_heads=n_heads, n_layers=n_layers, ffn_mult=ffn_mult,
            dropout=dropout, attn_dropout=attn_dropout,
            n_streams=n_streams, sinkhorn_tmax=sinkhorn_tmax, mhc_dropout=mhc_dropout
        )
    elif arch == "blocks_final_norm":
        m = FTTransformerBlocksFinalNorm(
            n_features=n_features, d_model=d_model, n_heads=n_heads, n_layers=n_layers,
            ffn_mult=ffn_mult, dropout=dropout, attn_dropout=attn_dropout
        )
    else:
        m = FTTransformerEncoder(
            n_features=n_features, d_model=d_model, n_heads=n_heads, n_layers=n_layers,
            ffn_mult=ffn_mult, dropout=dropout, attn_dropout=attn_dropout
        )
    return m, sd, arch

@torch.inference_mode()
def predict_proba_one_pack(X_raw: np.ndarray, pack: dict, batch_size=4096):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    use_amp = (device.type == "cuda")

    mu = pack.get("mu", np.zeros(X_raw.shape[1], np.float32))
    sig = pack.get("sig", np.ones (X_raw.shape[1], np.float32))
    Xn = apply_standardizer(X_raw, mu, sig)

    dl = DataLoader(
        XDataset(Xn),
        batch_size=int(batch_size),
        shuffle=False,
        num_workers=0,
        pin_memory=(device.type == "cuda"),
        drop_last=False
    )

    cfg = pack.get("cfg", {})
    if not isinstance(cfg, dict):
        cfg = {}
    m, sd, arch = build_model_for_pack(pack["state_dict"], X_raw.shape[1], cfg)
    m = m.to(device)

    try:
        m.load_state_dict(sd, strict=True)
    except Exception:
        missing, unexpected = m.load_state_dict(sd, strict=False)
        print(f"WARN load_state_dict strict=True failed -> strict=False used | arch={arch}")
        print(f"  missing={len(missing)} | unexpected={len(unexpected)}")
        if missing:
            print("  missing head:", missing[:8])
        if unexpected:
            print("  unexpected head:", unexpected[:8])

    m.eval()

    ps = []
    for xb in dl:
        xb = xb.to(device, non_blocking=True)
        with torch.cuda.amp.autocast(enabled=use_amp):
            logits = m(xb)
            p = torch.sigmoid(logits)
        ps.append(p.detach().cpu().numpy())
    return np.concatenate(ps, axis=0).astype(np.float32)

# ----------------------------
# 8) Gate inference (mean over packs) + OOM fallback
# ----------------------------
probs_list = []
t0 = time.time()

for i, pk in enumerate(packs):
    if not isinstance(pk, dict) or "state_dict" not in pk:
        raise ValueError(f"Pack {i} invalid / missing state_dict.")

    bs = int(BATCH_SIZE)
    while True:
        try:
            p = predict_proba_one_pack(X_test, pk, batch_size=bs)
            probs_list.append(p)
            break
        except RuntimeError as e:
            msg = str(e).lower()
            if ("out of memory" in msg) and torch.cuda.is_available() and bs > 256:
                torch.cuda.empty_cache()
                bs = max(256, bs // 2)
                print(f"OOM on pack {i} -> retry with batch_size={bs}")
                continue
            raise

    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()

gate_prob = np.mean(probs_list, axis=0).astype(np.float32)
print(f"OK — Gate inference done | rows={len(gate_prob)} | packs={len(probs_list)} | time={(time.time()-t0):.1f}s")

df_case = pd.DataFrame({
    "case_id": df_feat["case_id"].astype(str).values,
    "case_prob": gate_prob
})

# (AGG_MODE kept for compatibility; normally df_case already 1 row/case_id)
if AGG_MODE == "max":
    df_case = df_case.groupby("case_id", as_index=False)["case_prob"].max()
elif AGG_MODE == "p80":
    df_case = df_case.groupby("case_id", as_index=False)["case_prob"].quantile(0.80)
else:
    df_case = df_case.groupby("case_id", as_index=False)["case_prob"].mean()

df_case["is_forged"] = (df_case["case_prob"].values >= float(T_GATE)).astype(np.int32)

print("Gate summary:")
print("  AGG_MODE     :", AGG_MODE)
print("  T_GATE       :", T_GATE)
print("  forged_pred% :", float(df_case["is_forged"].mean()) * 100.0)

# ----------------------------
# 9) Optional pred_ens masks (if exists)
# ----------------------------
def find_pred_ens_dir():
    pref = Path("/kaggle/working/recodai_luc/cache/pred_ens")
    if pref.exists():
        return pref
    roots = [Path("/kaggle/working"), Path("/kaggle/input")]
    cands = []
    for r in roots:
        if not r.exists():
            continue
        try:
            cands.extend(list(r.glob("**/pred_ens")))
        except Exception:
            pass
    if not cands:
        return None
    cands.sort(key=lambda p: (("recodai_luc" not in str(p).lower()), ("cache" not in str(p).lower()), len(str(p))))
    return cands[0]

PRED_ENS_DIR = find_pred_ens_dir()
print("pred_ens dir:", (PRED_ENS_DIR if PRED_ENS_DIR else "(not found -> forged will fallback to authentic)"))

def rle_encode(mask_u8: np.ndarray):
    pixels = mask_u8.T.flatten()  # column-major
    dots = np.where(pixels == 1)[0]
    if len(dots) == 0:
        return "authentic"
    run_lengths = []
    prev = -2
    for b in dots:
        if b > prev + 1:
            run_lengths.extend((b + 1, 0))
        run_lengths[-1] += 1
        prev = b
    return json.dumps([int(x) for x in run_lengths])

_hw_cache = {}
def get_target_hw_from_test(case_id: str):
    if case_id in _hw_cache:
        return _hw_cache[case_id]
    p = TEST_IMG_DIR / f"{case_id}.png"
    if not p.exists():
        hits = list(TEST_IMG_DIR.glob(f"{case_id}.*"))
        if hits:
            p = hits[0]
    if not p.exists():
        _hw_cache[case_id] = None
        return None
    try:
        from PIL import Image
        im = Image.open(p)
        hw = (im.size[1], im.size[0])  # (H,W)
        _hw_cache[case_id] = hw
        return hw
    except Exception:
        _hw_cache[case_id] = None
        return None

def resize_mask_nearest(m: np.ndarray, target_hw):
    if target_hw is None:
        return m
    th, tw = int(target_hw[0]), int(target_hw[1])
    if m.shape[0] == th and m.shape[1] == tw:
        return m
    if _HAS_CV2:
        return cv2.resize(m.astype(np.uint8), (tw, th), interpolation=cv2.INTER_NEAREST).astype(np.uint8)
    from PIL import Image
    im = Image.fromarray(m.astype(np.uint8) * 255)
    im = im.resize((tw, th), resample=Image.NEAREST)
    return (np.array(im) > 127).astype(np.uint8)

def _try_unpack_mask_pack(pack: np.ndarray, hw):
    h, w = int(hw[0]), int(hw[1])
    pack = np.asarray(pack)
    if pack.dtype != np.uint8:
        pack = pack.astype(np.uint8, copy=False)
    for bitorder in ["little", "big"]:
        bits = np.unpackbits(pack, bitorder=bitorder)[: h*w]
        if bits.size == h*w:
            m = bits.reshape(h, w).astype(np.uint8)
            s = int(m.sum())
            if 0 < s < h*w:
                return m
    bits = np.unpackbits(pack)[: h*w]
    if bits.size == h*w:
        return bits.reshape(h, w).astype(np.uint8)
    return None

def load_mask_from_npz(npz_path: Path, target_hw=None):
    try:
        data = np.load(npz_path, allow_pickle=True)
    except Exception:
        return None, None

    keys = set(data.files)

    # direct rle string fields
    for k in ["annotation", "rle", "rle_str", "pred_rle"]:
        if k in keys:
            v = data[k]
            if isinstance(v, np.ndarray) and v.shape == ():
                v = v.item()
            if isinstance(v, bytes):
                v = v.decode("utf-8", errors="ignore")
            if isinstance(v, str) and len(v) > 0:
                return None, v

    # direct mask fields
    for k in ["mask", "mask_u8", "mask_bin", "pred_mask"]:
        if k in keys:
            m = data[k]
            if isinstance(m, np.ndarray):
                if m.ndim == 3:
                    m = m[..., 0]
                m = (m > 0.5).astype(np.uint8)
                m = resize_mask_nearest(m, target_hw)
                return m, None

    # packed mask bits
    if "mask_pack" in keys:
        pack = data["mask_pack"]
        hw = None
        for hk in ["hw", "shape", "mask_hw", "orig_hw", "resized_hw", "resized_hw_base", "HW"]:
            if hk in keys:
                v = data[hk]
                if isinstance(v, np.ndarray) and v.shape == ():
                    v = v.item()
                try:
                    vv = np.array(v).reshape(-1).tolist()
                    if len(vv) >= 2:
                        hw = (int(vv[0]), int(vv[1]))
                        break
                except Exception:
                    pass
        if hw is None and target_hw is not None:
            hw = target_hw
        if hw is not None:
            m = _try_unpack_mask_pack(pack, hw)
            if m is not None:
                m = resize_mask_nearest(m, target_hw)
                return m, None

    return None, None

def get_case_annotation(case_id: str, is_forged: int):
    if is_forged == 0:
        return "authentic"
    if PRED_ENS_DIR is None:
        return "authentic"

    cand = PRED_ENS_DIR / f"{case_id}.npz"
    if not cand.exists():
        hits = list(PRED_ENS_DIR.glob(f"{case_id}*.npz"))
        if not hits:
            return "authentic"
        hits.sort(key=lambda x: x.name)
        cand = hits[0]

    target_hw = get_target_hw_from_test(case_id)
    m, rle_direct = load_mask_from_npz(cand, target_hw=target_hw)
    if isinstance(rle_direct, str) and len(rle_direct) > 0:
        return rle_direct
    if m is None:
        return "authentic"
    return rle_encode(m)

# ----------------------------
# 10) Build submission (strict order as sample)
# ----------------------------
sub = pd.read_csv(SAMPLE_SUB)
if not {"case_id", "annotation"}.issubset(sub.columns):
    raise ValueError(f"sample_submission columns unexpected: {list(sub.columns)}")
sub["case_id"] = sub["case_id"].astype(str)

sub = sub.merge(df_case[["case_id", "case_prob", "is_forged"]], on="case_id", how="left")
sub["is_forged"] = sub["is_forged"].fillna(0).astype(int)

anns = [get_case_annotation(cid, int(fg)) for cid, fg in zip(sub["case_id"].values, sub["is_forged"].values)]
sub["annotation"] = anns
sub = sub[["case_id", "annotation"]]

out_path = Path("/kaggle/working/submission.csv")
sub.to_csv(out_path, index=False)

print("\nOK — submission saved:", out_path)
print("  rows       :", len(sub))
print("  authentic% :", float((sub["annotation"] == "authentic").mean()) * 100.0)
display(sub.head(10))


Using:
  FINAL_MODEL_PT: /kaggle/input/recod-ailuc-dinov2-train/recodai_luc_gate_artifacts/final_gate_model.pt
  BUNDLE_DIR    : /kaggle/input/recod-ailuc-dinov2-train/recodai_luc_gate_artifacts
  ART_DIR       : /kaggle/input/recod-ailuc-dinov2-train/recodai_luc_gate_artifacts
  COMP_ROOT   : /kaggle/input/recodai-luc-scientific-image-forgery-detection
  SAMPLE_SUB  : /kaggle/input/recodai-luc-scientific-image-forgery-detection/sample_submission.csv
  TEST_IMG_DIR: /kaggle/input/recodai-luc-scientific-image-forgery-detection/test_images
Artifacts:
  feature_cols: from final_gate_model.pt
  pack_source : blob.full_packs
  n_features  : 84
  n_packs     : 1
  thresholds  : /kaggle/input/recod-ailuc-dinov2-train/recodai_luc_gate_artifacts/model_bundle_v5_notebook3/thresholds.json
  T_GATE      : 0.07369999999999999
Base test cases: 1
Feature candidates:
  pred_features_test* : /kaggle/input/recod-ailuc-dinov2-train/recodai_luc_gate_artifacts/pred_features_test_cfg_bf03779a3773.csv
  matc

,case_id,annotation
0,45,authentic
